# Guanajuato WOFOST calibration with SIAP

This notebook reads the comparison CSV from `guanajuato_wofost_baseline.ipynb` when available. On a fresh checkout, it builds the same comparison from the configured GCS profile and SIAP files. Keep the crop, water mode, season dates, and other settings aligned with the baseline notebook when comparing their results. Dataverse is excluded because its yield workbook ends in 2022, before the 2024–2025 weather used here.

The full-data fitted error is an in-sample diagnostic. Leave-one-year-out (LOYO) predicts each year from the other year's factor. With two years, each fold trains on one year, so this is a limited temporal check. SIAP yield is municipal production per sown hectare; WOFOST dry-matter yield is simulated at one point. Moisture and spatial bases may differ.

In [ ]:
import sys
from pathlib import Path
repo_root = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents)
                  if (candidate / "gradientdescentfarmers" / "model_builder").is_dir()), None)
if repo_root is None:
    raise RuntimeError("Open this notebook from within the repository or install the package")
sys.path.insert(0, str(repo_root))

In [ ]:
import pandas as pd
from gradientdescentfarmers.model_builder.calibration import (
    calibrate_siap_yield, compare_siap_yield, error_metrics,
)

USE_SYNTHETIC_SMOKE = False
CROP = "maize"
VARIETY = "Grain_maize_202"
WATER_MODE = "potential"  # potential, rainfed, or irrigated
SEASONS = [("2024-05-15", "2024-12-15"), ("2025-05-15", "2025-12-15")]
RUN_URI = "gs://data_dump_samples/farm_profiles/guanajuato_valle_demo/runs/67b66942681a4214"
GCP_PROJECT = "gdfarming"
MUNICIPALITY = "Valle de Santiago"
SIAP_CROP_LABELS = {"maize": "Maíz grano"}
ELEVATION_OVERRIDE_M = None
CO2_PPM = 420.0

# Fill these measured values for water-limited scenarios, as in the baseline notebook.
SOIL_PARAMETERS = {"SMW": None, "SMFCF": None, "SM0": None, "RDMSOL": None,
                   "CRAIRC": None, "SOPE": None, "KSUB": None, "K0": None}
INITIAL_AVAILABLE_WATER_CM = None
IRRIGATION_SYSTEM = None
IRRIGATION_EVENTS_BY_YEAR = {2024: [], 2025: []}

scenario_tag = f"{WATER_MODE}_{CROP}"
comparison_name = (f"siap_synthetic_{scenario_tag}_comparison.csv"
                   if USE_SYNTHETIC_SMOKE else f"siap_{scenario_tag}_comparison.csv")
COMPARISON_CSV = repo_root / "data" / "model_builder" / comparison_name

if COMPARISON_CSV.exists():
    comparison_df = pd.read_csv(COMPARISON_CSV)
else:
    from gradientdescentfarmers.model_builder.gcs import GCSDataClient
    from gradientdescentfarmers.model_builder.wofost import run_wofost

    if USE_SYNTHETIC_SMOKE:
        # Invented weather and elevation are only for a wiring check.
        dates = pd.date_range("2024-01-01", "2025-12-31")
        weather = pd.DataFrame({"date": dates, "tmin_c": 12.0, "tmax_c": 27.0,
                                "precip_mm": 2.0, "rh_pct": 65.0, "wind_m_s": 2.0,
                                "solar_mj_m2_day": 19.0})
        site = {"latitude": 20.4243195, "longitude": -101.1466639}
        elevation = 1700.0
        run_id = "SYNTHETIC_SMOKE"
    else:
        gcs = GCSDataClient(project=GCP_PROJECT)
        profile = gcs.load_profile(RUN_URI)
        weather = profile.table("weather_daily")
        site = profile.manifest["farmer_inputs"]
        elevation = (ELEVATION_OVERRIDE_M if ELEVATION_OVERRIDE_M is not None
                     else profile.manifest.get("topography", {}).get("elevation_m"))
        if elevation is None:
            raise ValueError("Supply a measured elevation when profile topography is missing")
        run_id = profile.manifest["run_id"]

    simulation_df = pd.concat([
        run_wofost(
            weather, crop=CROP, variety=VARIETY, sowing_date=start, end_date=end,
            latitude=site["latitude"], longitude=site["longitude"],
            elevation_m=float(elevation), co2_ppm=CO2_PPM, run_id=run_id,
            water_mode=WATER_MODE,
            soil_parameters=None if WATER_MODE == "potential" else SOIL_PARAMETERS,
            initial_available_water_cm=(None if WATER_MODE == "potential"
                                        else INITIAL_AVAILABLE_WATER_CM),
            irrigation_system=IRRIGATION_SYSTEM if WATER_MODE == "irrigated" else None,
            irrigation_events=(IRRIGATION_EVENTS_BY_YEAR.get(pd.Timestamp(start).year, [])
                               if WATER_MODE == "irrigated" else None),
        )
        for start, end in SEASONS
    ], ignore_index=True)
    if USE_SYNTHETIC_SMOKE:
        finals = simulation_df.loc[simulation_df.is_final_day, ["year", "yield_t_ha"]]
        siap_df = pd.DataFrame({"year": finals.year, "municipality": "TEST",
                                "crop": SIAP_CROP_LABELS[CROP], "unit": "Tonelada",
                                "sown_ha": 10.0,
                                "production_volume": finals.yield_t_ha.astype(float) * 5})
        municipality_for_check = "TEST"
    else:
        siap_prefix = "gs://data_dump_samples/SIAP_Agricultural_Production_2000_2025/"
        siap_uris = [f"{siap_prefix}Municipal (2003_2025)/Cierre_agricola_mun_{year}.csv"
                     for year in sorted(simulation_df.loc[simulation_df.is_final_day, "year"].unique())]
        siap_df = pd.concat([gcs.read_frame(uri) for uri in siap_uris], ignore_index=True)
        municipality_for_check = MUNICIPALITY
    comparison_df = compare_siap_yield(
        simulation_df, siap_df, municipality=municipality_for_check,
        crop_labels=SIAP_CROP_LABELS,
    )
    COMPARISON_CSV.parent.mkdir(parents=True, exist_ok=True)
    comparison_df.to_csv(COMPARISON_CSV, index=False)
    print("Built missing baseline comparison:", COMPARISON_CSV)

expected_years = {pd.Timestamp(end).year for _, end in SEASONS}
assert set(comparison_df.year) == expected_years, "Comparison years do not match SEASONS"
assert set(comparison_df.crop) == {CROP}, "Comparison file contains a different crop"
assert set(comparison_df.water_mode) == {WATER_MODE}, "Comparison file contains a different water mode"
print("Paired municipality and years:", comparison_df.municipality.unique(), sorted(comparison_df.year.unique()))

In [ ]:
calibration_df = calibrate_siap_yield(comparison_df)
error_summary_df = pd.concat([
    error_metrics(calibration_df, prediction_column=column)
    for column in ("simulated_t_ha", "calibrated_t_ha", "loyo_t_ha")
], ignore_index=True)
error_summary_df.insert(0, "crop", CROP)
error_summary_df.insert(1, "water_mode", WATER_MODE)
display(calibration_df[["year", "crop", "water_mode", "observed_t_ha", "simulated_t_ha",
                        "calibration_factor", "calibrated_t_ha", "loyo_factor",
                        "loyo_t_ha", "raw_error_t_ha", "fit_error_t_ha", "loyo_error_t_ha"]])
display(error_summary_df)

In [ ]:
output_name = f"siap_synthetic_{scenario_tag}_calibration.csv" if USE_SYNTHETIC_SMOKE else f"siap_{scenario_tag}_calibration.csv"
OUTPUT_CSV = repo_root / "data" / "model_builder" / output_name
calibration_df.to_csv(OUTPUT_CSV, index=False)
print("Saved:", OUTPUT_CSV)